# 📗 [참고] 다중공선성 — 독립변수끼리 얽힐 때

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

**이 노트북은 참고 자료입니다.** 9일차 본 교안은 **가설검정(교안 01)** 과 **회귀분석(교안 02)** 입니다. 교안 02 의 다중회귀에서 **배기량 계수가 유의하지 않게 나오는 장면**을 봤는데, "왜 그런가 · 얼마나 심한가 · 어떻게 처리하나"까지 파고드는 것이 이 노트북입니다. 본 교안·과제를 마친 뒤 이어서 보면 됩니다.

## 이 노트북의 구성
| 파트 | 내용 |
|---|---|
| **1부 증상** | 혼자면 강한 변수가 함께 넣으면 사라지는 현상 · 왜 생기나 · 무엇이 문제인가(표준오차 팽창) |
| **2부 진단** | 상관행렬 · **VIF**(분산팽창인자) · `Cond. No.` — 얼마나 얽혔는지 재는 세 도구 |
| **3부 대처** | 목적(예측/해석) 먼저 묻기 · 변수 제거 · 변수 결합 · 중심화 · 정규화 회귀 |
| **실습 문제 (2문)** | 고객센터 데이터로 **진단→처리**, 펭귄 데이터로 **파생변수가 만든 공선성** 다루기 |

## 풀이 방법
1. 교안 파트는 **🖐️ 함께 따라하기** 셀을 직접 채우며 읽습니다.
2. 실습 문제는 각 단계의 **답안 셀**(`# 여기에 코드를 작성하세요`)을 채우고, 아래 **자가채점 셀**로 확인합니다.
3. **판단·서술 단계**는 정해진 답이 없습니다 — 정답 노트북의 모범 서술과 비교하세요.

화이팅!

> 🔧 **이 단원의 도구**: 회귀는 교안 02 와 똑같이 **`statsmodels`**(`smf.ols`)를 씁니다. VIF 는 **보조 회귀**(한 변수를 나머지 변수들로 회귀)의 결정계수로 직접 계산합니다 — `1 / (1 - R²)` 한 줄이면 되고, 이 식이 곧 VIF 의 정의라 계산 과정 자체가 개념 설명이 됩니다.

In [ ]:
# [제공 코드] 회귀·시각화에 쓸 라이브러리와 한글 폰트를 준비합니다.
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf          # 회귀분석 (smf.ols)

import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지
sns.set_theme(font=KOREAN_FONT, rc={'axes.unicode_minus': False})

---
# 1부. 증상 — 혼자면 강한데, 함께 넣으면 사라진다

## 1. 교안 02 에서 본 장면

교안 02 의 다중회귀에서 이런 일이 있었습니다. 배기량(`displacement`)은 **혼자 넣으면** 연비를 아주 잘 설명하는데, 무게·마력과 **함께 넣으면** 계수가 유의하지 않게 나왔습니다. 변수를 더 넣었을 뿐인데 왜 힘을 잃을까요? 직접 두 모형을 나란히 적합해 봅니다.

In [ ]:
df = pd.read_csv('data/mpg.csv')
m = df.dropna(subset=['mpg', 'weight', 'horsepower', 'displacement'])
print('분석 대상:', len(m), '대  (교안 02 와 같은 자동차 연비 데이터)')
display(m[['mpg', 'weight', 'horsepower', 'displacement']].head())

# ① 배기량 하나만 넣은 단순회귀
solo = smf.ols('mpg ~ displacement', data=m).fit()
fmt_solo = '[단독] 배기량 계수 = %.4f, p = %.3g, R^2 = %.4f'
print(fmt_solo % (solo.params['displacement'], solo.pvalues['displacement'], solo.rsquared))

# ② 무게·마력과 함께 넣은 다중회귀
multi = smf.ols('mpg ~ weight + horsepower + displacement', data=m).fit()
fmt_multi = '[함께] 배기량 계수 = %.4f, p = %.4f  <- 0.05 보다 훨씬 크다(유의하지 않음)'
print(fmt_multi % (multi.params['displacement'], multi.pvalues['displacement']))
print()
fmt_gap = '배기량 혼자서도 연비 변동의 %.0f%% 를 설명했는데, 함께 넣으니 p 가 %.2f 로 올라갔다.'
print(fmt_gap % (solo.rsquared * 100, multi.pvalues['displacement']))

## 2. 왜 그럴까 — 같은 정보를 담은 변수들

답은 **독립변수끼리의 상관**에 있습니다. 무거운 차는 대개 엔진도 크고 마력도 셉니다. 세 변수가 사실상 "차가 크다"는 **같은 이야기**를 조금씩 다르게 하고 있는 것입니다.

다중회귀의 계수는 "**다른 변수를 고정한 채** 그 변수만 1 늘었을 때의 효과"였습니다. 그런데 무게·마력이 고정되면 배기량은 거의 움직일 여지가 없습니다. **혼자 설명할 몫이 남지 않으니** 계수를 정확히 추정할 근거가 사라집니다.

이렇게 독립변수끼리 강하게 상관된 상태를 **다중공선성(multicollinearity)** 이라고 부릅니다. 먼저 상관행렬로 눈으로 확인해 봅시다.

In [ ]:
# 독립변수끼리의 상관 — 종속변수(mpg)가 아니라 '설명하는 쪽'끼리를 본다
cols = ['mpg', 'weight', 'horsepower', 'displacement']
corr = m[cols].corr()
display(corr.round(3))

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title('상관행렬 — 독립변수 세 개가 서로 0.86 이상으로 얽혀 있다')
plt.show()

print('무게 - 배기량 상관 = %.3f  (거의 같은 정보)' % corr.loc['weight', 'displacement'])
print('무게 - 마력   상관 = %.3f' % corr.loc['weight', 'horsepower'])
print('마력 - 배기량 상관 = %.3f' % corr.loc['horsepower', 'displacement'])

## 3. 다중공선성은 어디서 생기나 — 네 가지 전형

| 원인 | 예 | 특징 |
|---|---|---|
| **원래 겹치는 변수** | 무게·배기량·마력 · 키·몸무게 · 매출·주문수 | 가장 흔합니다. 도메인상 같이 커지는 값들 |
| **파생·합계·비율 변수** | 국어+영어+수학 과 총점을 같이 넣기 · 길이×깊이(면적)를 원래 두 변수와 같이 넣기 | 부분과 전체를 함께 넣으면 **정의상** 겹칩니다 |
| **제곱항·상호작용항** | `x` 와 `x²` · `x1` 과 `x1×x2` | **구조적 공선성** — 값이 커질수록 제곱도 커지니 당연히 상관됩니다 |
| **더미 함정** | 범주 3개를 더미 3개로 **전부** 넣고 절편도 둠 | 세 열의 합이 늘 1 이라 절편과 완전히 겹칩니다(교안 02 참고) |

> 앞의 둘은 **데이터가 원래 그런 것**이고, 뒤의 둘은 **내가 만들어 낸 것**입니다. 만들어 낸 공선성은 **중심화**로 상당 부분 없앨 수 있습니다(3부).

## 4. 무엇이 문제인가 — "분산팽창"이라는 이름의 뜻

다중공선성이 있으면 계수 자체가 틀리는 것이 아닙니다. **계수가 흔들립니다** — 정확히 말하면 **계수의 표준오차(std err)가 부풀려집니다.** 그 결과가 이렇게 이어집니다.

> 표준오차 커짐 → **신뢰구간이 넓어짐** → **p-value 가 커짐**(유의하지 않게 보임) → 표본이 조금만 달라져도 **계수의 크기·부호가 요동**

얼마나 부풀려지는지를 재는 값이 다음 절의 **VIF** 이고, 이름 그대로 **분산팽창인자**입니다. **계수의 표준오차는 겹침이 전혀 없을 때보다 정확히 √VIF 배**가 됩니다. 숫자로 확인해 봅시다.

In [ ]:
# 겹침이 전혀 없었다면 표준오차가 얼마였을지와, 실제 표준오차를 비교한다
#   겹침이 없을 때의 표준오차 = 잔차표준편차 / (그 변수의 표준편차 x sqrt(n-1))
n = len(m)
sigma = np.sqrt(multi.mse_resid)          # 잔차의 표준편차

fmt_se = '  %-13s 실제 표준오차 = %.6f, 겹침이 없었다면 = %.6f  ->  %.2f 배로 부풀려짐'
for name in ['weight', 'horsepower', 'displacement']:
    se_real = multi.bse[name]
    se_ideal = sigma / (m[name].std() * np.sqrt(n - 1))
    print(fmt_se % (name, se_real, se_ideal, se_real / se_ideal))

print()
print('배기량은 표준오차가 3.2 배로 부풀려졌다 -> 신뢰구간이 3.2 배 넓어졌다는 뜻이다.')
print('계수(-0.0058)가 틀린 것이 아니라, 그 값을 믿을 만한 폭이 너무 넓어진 것이다.')

> ### 중요 — 예측은 멀쩡합니다
> 다중공선성이 있어도 **모형 전체의 R²·예측값·잔차는 아무 문제가 없습니다.** 망가지는 것은 **"각 변수가 몇 점씩 기여했나"를 나누는 일**뿐입니다. 그래서 **목적이 예측이면 다중공선성은 손대지 않아도 되는 경우가 많습니다**(3부에서 다시 다룹니다).

### 🖐️ 함께 따라하기 — 겹치는 변수를 하나 넣어 보기
데모는 **자동차 연비(mpg)** 였습니다. 따라하기는 **고객센터 상담 기록**(`callcenter_calls.csv`)으로 같은 현상을 만들어 봅니다.

만족도를 **대기시간·상담시간·상담원경력** 세 변수로 설명하는 모형에, **대기고객수**(대기 줄에 서 있던 인원)를 하나 더 넣습니다. 줄이 길면 오래 기다리니 대기시간과 사실상 같은 정보입니다. 넣기 전후로 **대기시간 계수의 표준오차·신뢰구간·p-value** 가 어떻게 달라지는지 보세요.

> ⚠️ 이 셀에서 만드는 `cc` 를 **이후 따라하기와 실습 문제에서 계속 씁니다** — 꼭 실행하고 넘어가세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# ※ 데모는 '자동차 연비'였죠. 이번엔 다른 데이터(고객센터 상담 기록)로 연습합니다.
# 1) data/callcenter_calls.csv 를 cc 로 읽고, 회귀식에 쓰기 쉽게 영문 별칭 열을 만든다
#    sat(만족도) · wait(대기시간_초) · talk(상담시간_분) · exp(상담원경력_년) · queue(대기고객수)
# 2) 대기시간과 대기고객수의 상관(cc['wait'].corr(cc['queue']))을 출력한다
# 3) m3 = smf.ols('sat ~ wait + talk + exp', data=cc).fit() 로 3변수 모형을 적합한다
# 4) m4 = 여기에 + queue 를 더한 4변수 모형을 적합한다
# 5) 두 모형에서 wait 의 계수·표준오차(.bse)·p-value 를 나란히 출력해 비교한다
# 6) .conf_int().loc['wait'] 로 두 모형의 신뢰구간을 출력하고, 표준오차가 몇 배가 됐는지 계산한다

### ✅ 바로 확인 퀴즈
**1.** 다중공선성이 있으면 회귀 결과의 무엇이 망가지고, 무엇은 멀쩡한가요?

<details><summary>정답 보기</summary>

**계수의 표준오차가 부풀려져** 신뢰구간이 넓어지고 p-value 가 커집니다(계수 해석이 흔들림). 반면 **모형 전체의 R²·예측값**은 영향을 받지 않습니다. 그래서 목적이 예측이면 문제가 되지 않을 수 있습니다.

</details>

**2.** 어떤 변수가 다중회귀에서 유의하지 않게 나왔습니다. "이 변수는 종속변수와 관계가 없다"고 결론지어도 될까요?

<details><summary>정답 보기</summary>

안 됩니다. 배기량처럼 **혼자 넣으면 아주 강한 변수**도 겹치는 변수와 함께 넣으면 유의하지 않게 보일 수 있습니다. "관계가 없다"가 아니라 "**다른 변수들이 이미 설명한 뒤에 추가로 설명할 몫이 적다**"로 읽어야 합니다.

</details>

---
# 2부. 진단 — 얼마나 얽혔는지 재는 세 가지

## 5. 상관행렬 — 빠르지만 한계가 있다

가장 손쉬운 진단은 앞에서 본 **독립변수끼리의 상관행렬**입니다. 절댓값 0.8 을 넘는 짝이 보이면 일단 의심합니다.

다만 상관행렬은 **두 변수씩 짝지어서만** 봅니다. 그래서 **셋 이상이 합쳐져 만드는 겹침**은 놓칩니다 — 예를 들어 `총점 = 국어 + 영어 + 수학` 이면 총점과 각 과목의 상관은 0.6 정도로 평범해 보이지만, 세 과목을 합치면 총점이 **완전히** 결정됩니다. 그래서 다음 지표가 필요합니다.

## 6. VIF (분산팽창인자) — 표준 진단 도구

**VIF** 는 "그 변수를 **나머지 독립변수 전부로** 설명하면 얼마나 설명되는가"를 재기 때문에 상관행렬의 한계를 넘습니다.

$$ VIF_j = \frac{1}{1 - R_j^2} $$

여기서 $R_j^2$ 는 **변수 j 를 종속변수로 두고 나머지 변수들로 회귀**한 보조 모형의 결정계수입니다. 다른 변수들로 그 변수가 잘 설명될수록($R_j^2$ 이 클수록) VIF 가 커집니다.

| 다른 변수들의 설명력 $R_j^2$ | VIF | 표준오차 팽창(√VIF) | 판정 |
|---|---|---|---|
| 0.00 | 1.0 | 1.00 배 | 완전히 독립 |
| 0.50 | 2.0 | 1.41 배 | 문제 없음 |
| 0.80 | **5.0** | 2.24 배 | **경고선** — 계수 해석에 주의 |
| 0.90 | **10.0** | 3.16 배 | **강한 컷오프** — 변수 정리를 적극 고려 |
| 0.99 | 100.0 | 10.0 배 | 사실상 같은 변수 |

> 기준선 5·10 의 근거가 이 표에 있습니다. **다른 변수들이 그 변수를 90% 설명하면 VIF 는 10** 입니다.

In [ ]:
# VIF 계산 — 각 변수를 '나머지 변수들'로 회귀해 1/(1-R^2)
def vif_table(data, predictors):
    rows = []
    for name in predictors:
        others = [p for p in predictors if p != name]
        r2_aux = smf.ols(name + ' ~ ' + ' + '.join(others), data=data).fit().rsquared
        rows.append({'변수': name, 'R2_보조회귀': round(r2_aux, 3),
                     'VIF': round(1 / (1 - r2_aux), 3)})
    return pd.DataFrame(rows)

vif_mpg = vif_table(m, ['weight', 'horsepower', 'displacement'])
display(vif_mpg)
print('세 변수 모두 경고선 5 를 넘고, 배기량은 강한 컷오프 10 도 넘는다 -> 심한 다중공선성')

# 막대그래프로 기준선과 함께 보기
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=vif_mpg, x='변수', y='VIF', ax=ax, color='steelblue')
ax.axhline(5, color='orange', linestyle='--', label='경고선 5')
ax.axhline(10, color='red', linestyle='--', label='강한 컷오프 10')
ax.set_title('변수별 VIF — 기준선과 비교')
ax.legend()
plt.show()

## 7. `Cond. No.` — summary 맨 아래 그 숫자

교안 02 에서 `.summary()` 를 읽을 때 **"지금은 무시하세요"** 로 넘겼던 `Cond. No.`(조건수)가 바로 이 이야기입니다. VIF 가 **변수마다 하나씩** 나오는 반면, `Cond. No.` 는 **설계행렬 전체의 얽힘을 숫자 하나로** 요약합니다.

- 보통 **30 이상**이면 공선성을 의심하고, 수백~수천이면 강한 신호로 봅니다.
- 다만 이 값은 **변수의 단위·크기(스케일)에 민감**합니다. 무게(수천)와 마력(수백)처럼 자릿수가 다르면 겹침이 없어도 커집니다. 그래서 실무 진단은 **VIF 를 주로 쓰고**, `Cond. No.` 는 참고로 봅니다.

In [ ]:
# 세 변수 모형 vs 배기량을 뺀 두 변수 모형의 조건수
two = smf.ols('mpg ~ weight + horsepower', data=m).fit()
print('3변수 모형 Cond. No. = %.0f' % multi.condition_number)
print('2변수 모형 Cond. No. = %.0f' % two.condition_number)
print('겹치는 변수를 빼면 줄어들지만, 단위 차이 때문에 여전히 큰 값이다.')
print('-> 판단은 VIF 로 한다. Cond. No. 는 참고 지표.')

### 🖐️ 함께 따라하기 — 고객센터 모형의 VIF
앞 따라하기에서 만든 `cc` 로, **대기고객수를 넣은 4변수 모형**의 VIF 를 구해 보세요. 위에서 만든 `vif_table` 함수를 그대로 쓰면 됩니다.

그리고 **표준오차가 √VIF 배 부풀려진다**는 규칙이 실제로 맞는지 직접 확인해 보세요 — 3변수 모형에서 `wait` 의 VIF 는 거의 1 이므로, 두 모형의 표준오차 비가 √VIF 와 같아야 합니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) vif_table(cc, ['wait', 'talk', 'exp', 'queue']) 로 VIF 표를 만들어 display 한다
# 2) 표에서 wait 의 VIF 를 꺼내 출력한다 (기준선 5·10 과 비교해 한 줄 설명도 함께)
# 3) m4.bse['wait'] / m3.bse['wait'] 로 표준오차가 몇 배가 됐는지 구한다
# 4) 그 값이 np.sqrt(wait 의 VIF) 와 같은지 나란히 출력해 확인한다

---
# 3부. 대처 — 어떻게 처리하나

## 8. 손대기 전에 먼저 물을 것: 목적이 무엇인가

VIF 가 10 을 넘었다고 **반드시** 변수를 지워야 하는 것은 아닙니다. 먼저 이렇게 묻습니다.

| 목적 | 다중공선성이 문제인가 | 해야 할 일 |
|---|---|---|
| **예측** — 앞으로 들어올 데이터의 y 를 맞히는 것이 목표 | **대개 문제 아님** (예측값·R² 는 멀쩡) | 그대로 두어도 됩니다 |
| **해석** — 어느 변수가 얼마나 기여하는지 말해야 함 | **문제** (계수·p 를 믿기 어려움) | 아래 방법으로 정리합니다 |
| **관심 변수는 따로 있음** — 겹치는 것들은 통제변수일 뿐 | 관심 변수의 VIF 만 낮으면 **문제 아님** | 관심 변수의 VIF 만 확인합니다 |

> "VIF 10 초과 = 무조건 삭제" 는 실무에서 흔한 오해입니다. **무엇을 말하려고 이 회귀를 돌리는가**가 먼저입니다.

## 9. 방법 ① 겹치는 변수 중 하나를 뺀다 — 가장 단순하고 흔한 해법

겹치는 변수 중 **도메인에서 더 의미 있고, 측정이 정확하고, 종속변수와 더 강한** 것을 남깁니다. 여기서는 VIF 가 가장 큰 **배기량**을 빼 봅니다. 무엇이 좋아지고 무엇이 그대로인지 보세요.

In [ ]:
# 배기량을 뺀 모형과 원래 모형 비교
vif_two = vif_table(m, ['weight', 'horsepower'])
display(vif_two)

fmt_cmp = '%-18s Adj R^2 = %.4f | weight 계수 = %.5f (se %.5f, p=%.4f) | horsepower 계수 = %.4f (se %.5f, p=%.4f)'
print(fmt_cmp % ('[3변수 원래 모형]', multi.rsquared_adj,
                 multi.params['weight'], multi.bse['weight'], multi.pvalues['weight'],
                 multi.params['horsepower'], multi.bse['horsepower'], multi.pvalues['horsepower']))
print(fmt_cmp % ('[배기량 제거]', two.rsquared_adj,
                 two.params['weight'], two.bse['weight'], two.pvalues['weight'],
                 two.params['horsepower'], two.bse['horsepower'], two.pvalues['horsepower']))
print()
print('VIF 10.31 -> 3.96 으로 내려갔고, 두 계수의 표준오차도 함께 줄었다.')
fmt_keep = '설명력은 Adj R^2 %.4f -> %.4f 로 사실상 그대로다 -> 배기량은 빼도 잃는 것이 없었다.'
print(fmt_keep % (multi.rsquared_adj, two.rsquared_adj))

## 10. 방법 ② 겹치는 변수들을 하나로 합친다

"엔진이 크다"는 하나의 개념을 마력·배기량 두 열이 나눠 담고 있다면, **표준화해서 더해 하나의 지표**로 만들 수 있습니다(합성 변수). 개념이 하나로 유지되고 정보도 버리지 않는다는 장점이 있습니다.

다만 **항상 더 낫지는 않습니다.** 아래에서 직접 해 보면, 합친 지표가 여전히 무게와 강하게 상관되어 VIF 가 기준선 5 를 넘습니다. **어느 방법이 나은지는 데이터로 확인해야** 합니다.

In [ ]:
# 마력·배기량을 표준화해 더한 '엔진 크기' 지표로 합치기
m = m.copy()
m['engine'] = ((m['horsepower'] - m['horsepower'].mean()) / m['horsepower'].std()
               + (m['displacement'] - m['displacement'].mean()) / m['displacement'].std())

combined = smf.ols('mpg ~ weight + engine', data=m).fit()
vif_combined = vif_table(m, ['weight', 'engine'])
display(vif_combined)

fmt_res = '  %-16s Adj R^2 = %.4f, 최대 VIF = %.2f'
print(fmt_res % ('배기량 제거', two.rsquared_adj, vif_two['VIF'].max()))
print(fmt_res % ('마력+배기량 합성', combined.rsquared_adj, vif_combined['VIF'].max()))
print()
print('이 데이터에서는 합성보다 제거가 더 깔끔했다(VIF 3.96 vs 6.74, 설명력은 동일).')
print('합성 지표도 결국 무게와 강하게 상관되기 때문이다.')

## 11. 방법 ③ 중심화 — 내가 만들어 낸 공선성 없애기

**제곱항·상호작용항**을 넣으면 공선성이 거의 반드시 생깁니다. `weight` 가 크면 `weight²` 도 크니 당연히 상관되죠. 이건 데이터의 성질이 아니라 **내가 항을 만들면서 생긴 구조적 문제**라, **평균을 빼서(중심화) 만들면** 대부분 사라집니다.

핵심은 **중심화해도 모형의 예측·설명력은 전혀 달라지지 않는다**는 점입니다. 변수의 원점을 옮겼을 뿐이니까요. 직접 확인해 봅시다.

In [ ]:
# 제곱항을 원래 값으로 넣을 때 vs 중심화한 값으로 넣을 때
m['w2'] = m['weight'] ** 2
m['weight_c'] = m['weight'] - m['weight'].mean()      # 중심화(평균 빼기)
m['w2_c'] = m['weight_c'] ** 2

raw = smf.ols('mpg ~ weight + w2', data=m).fit()
cen = smf.ols('mpg ~ weight_c + w2_c', data=m).fit()

print('[원래 값]  ', vif_table(m, ['weight', 'w2'])['VIF'].tolist())
print('[중심화]   ', vif_table(m, ['weight_c', 'w2_c'])['VIF'].tolist())
print()
print('Adj R^2 : 원래 %.4f / 중심화 %.4f' % (raw.rsquared_adj, cen.rsquared_adj))
print('예측값이 완전히 같은가? ', np.allclose(raw.fittedvalues, cen.fittedvalues))
print()
print('VIF 62.9 -> 1.29 로 내려갔는데 설명력과 예측은 한 치도 달라지지 않았다.')
print('공짜로 얻는 개선이므로, 제곱항·상호작용항을 넣을 때는 습관적으로 중심화한다.')

## 12. 방법 ④·⑤ — 표본을 늘린다 · 정규화 회귀

- **표본을 늘린다**: 표준오차는 표본이 커지면 줄어듭니다. 공선성이 있어도 데이터가 충분히 많으면 계수를 그럭저럭 추정할 수 있습니다(가능한 경우가 많지는 않지만, 원리는 알아 둘 만합니다).
- **정규화 회귀(릿지·라쏘)**: 계수의 크기에 벌점을 주어 흔들림을 잡는 방법입니다. **이 단원 범위를 넘습니다** — 머신러닝 단원에서 다시 만납니다. 이름과 쓰임만 알아 두세요.

## 13. 정리 — 무엇을 언제 쓰나

| 상황 | 권하는 방법 |
|---|---|
| 목적이 **예측**이다 | 그대로 둔다 (예측 성능에는 영향이 없다) |
| 겹치는 변수 중 하나가 **분명히 더 의미 있다** | 나머지를 **제거**한다 |
| 겹치는 변수들이 **하나의 개념**을 나눠 담고 있다 | **합성 지표**로 묶는다 |
| **제곱항·상호작용항** 때문에 생겼다 | **중심화**한다 (예측은 그대로) |
| 어느 변수도 버릴 수 없고 해석도 해야 한다 | 표본 확보 · 정규화 회귀(머신러닝 단원) |

> 어떤 방법을 쓰든 **처리 전후의 VIF·계수·Adj R² 를 나란히 놓고 확인**하는 것이 마지막 단계입니다. 이 노트북의 코드가 전부 그 형식이었던 이유입니다.

## 14. 리포트에 어떻게 쓰나

> "무게·마력·배기량을 함께 넣은 모형에서 배기량의 계수는 유의하지 않았습니다(p = 0.381). 다만 이는 배기량이 연비와 무관해서가 아니라, **세 변수의 VIF 가 5~10 을 넘을 만큼 서로 얽혀 있어**(배기량 VIF = 10.31) 각 변수의 몫을 나누기 어렵기 때문입니다. 배기량을 제외해도 설명력은 Adj R² 0.7047 → 0.7049 로 유지되므로, **무게·마력 두 변수 모형**으로 보고합니다."

핵심은 **"유의하지 않다"로 끝내지 않는 것**입니다. 왜 그렇게 보이는지(공선성), 어떻게 처리했는지, 처리 후 무엇이 달라졌는지까지 적어야 읽는 사람이 판단할 수 있습니다.

### ✅ 바로 확인 퀴즈
**1.** VIF 가 12 인 변수가 있습니다. 무조건 지워야 하나요?

<details><summary>정답 보기</summary>

아닙니다. **목적을 먼저 봅니다.** 예측이 목적이면 그대로 두어도 되고, 그 변수가 관심 대상이 아닌 통제변수일 뿐이라면 관심 변수의 VIF 만 낮으면 됩니다. 해석이 목적이고 그 변수를 말해야 할 때 제거·합성·중심화를 검토합니다.

</details>

**2.** `x` 와 `x²` 를 함께 넣었더니 VIF 가 60 이 넘었습니다. 어떻게 하면 좋을까요?

<details><summary>정답 보기</summary>

**중심화**합니다 — `x` 에서 평균을 뺀 값으로 `x` 와 `x²` 를 다시 만듭니다. 이런 구조적 공선성은 중심화로 대부분 사라지고, **예측값과 설명력(R²)은 전혀 달라지지 않습니다.**

</details>

---
# 실습 문제 — 직접 진단하고 처리하기

두 문제 모두 **진단 → 처리 → 판단** 흐름을 따릅니다. 각 단계의 답안 셀을 채우고 자가채점 셀로 확인하세요.

## 실습 문제 1 — 고객센터: 겹치는 변수를 정리하라
**배경**: 고객센터장이 "만족도를 좌우하는 요인을 순서대로 알려 달라"고 했습니다. 대기시간·상담시간·상담원경력·대기고객수 네 변수로 모형을 세웠는데, **대기시간과 대기고객수가 거의 같은 정보**라는 지적을 받았습니다. 진단하고 정리한 뒤 보고 문장을 만드세요.

> 1단계에서 데이터를 **새로 불러와** 같은 `cc` 로 끝까지 이어 갑니다. VIF 계산은 교안 파트에서 만든 `vif_table` 함수를 그대로 쓰면 됩니다.

**최종 목표(자가채점 기준)**
| 단계 | 확인 항목 |
| --- | --- |
| 1단계 | 4변수 모형 — `wait` p **0.0033**, `queue` p **0.7228**, Adj R² **0.5368** |
| 2단계 | VIF — `wait` **53.91**, `queue` **53.90** |
| 3단계 | `queue` 제거 후 — Adj R² **0.5375**, `wait` VIF **1.00** |
| 4단계 | 표준오차 팽창 배수 **7.34** = √VIF 확인 |
| 5단계 | 판단 서술 (자가채점 없음) |

### 1단계 — 네 변수 모형의 계수 읽기
**요구사항**:
- `data/callcenter_calls.csv` 를 `cc` 로 불러오고, 회귀식에 쓸 영문 별칭 열을 만드세요 — `sat`(만족도)·`wait`(대기시간_초)·`talk`(상담시간_분)·`exp`(상담원경력_년)·`queue`(대기고객수).
- `smf.ols('sat ~ wait + talk + exp + queue', data=cc).fit()` 로 적합해 `mod4` 에 담으세요.
- `wait` 의 p-value 를 `p_wait`, `queue` 의 p-value 를 `p_queue`, 조정 결정계수를 `adj4` 에 담으세요.
- 세 값을 출력하고, **어느 변수가 유의하지 않은지** 한 줄로 함께 출력하세요.
- p-value 는 소수 **넷째 자리**, Adj R² 는 소수 **셋째 자리**까지 비교합니다.

**예시**
```
round(p_wait, 4)  → 0.0033
round(p_queue, 4) → 0.7228
round(adj4, 3)    → 0.537
```

<details><summary>힌트</summary>

```text
접근방법:
- 데이터를 읽어 영문 별칭 열을 만든 뒤, 네 변수를 모두 넣은 모형을 적합한다.
- pvalues 와 rsquared_adj 에서 값을 꺼낸다.

세부구현:
1. read_csv 로 데이터를 cc 에 담고 sat·wait·talk·exp·queue 별칭 열을 만든다
2. smf.ols 로 sat ~ wait + talk + exp + queue 를 적합해 mod4 에 담는다
3. mod4.pvalues['wait'] · mod4.pvalues['queue'] 를 각각 담는다
4. mod4.rsquared_adj 를 adj4 에 담고 세 값을 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(p_wait - 0.0033) < 0.001
assert abs(p_queue - 0.7228) < 0.01
assert abs(adj4 - 0.5368) < 0.01
print("✅ 1단계 통과!")

### 2단계 — VIF 로 진단하기
**요구사항**:
- 네 변수 `['wait', 'talk', 'exp', 'queue']` 의 VIF 를 구해 `vif4`(DataFrame)에 담고 `display` 하세요 (교안 파트에서 만든 `vif_table` 함수를 써도 되고, 직접 for 문으로 계산해도 됩니다).
- `wait` 의 VIF 를 `vif_wait`, `queue` 의 VIF 를 `vif_queue` 에 담아 출력하세요.
- VIF 는 소수 **둘째 자리**까지 비교합니다.

**예시**
```
round(vif_wait, 2)  → 53.91    # 강한 컷오프 10 을 크게 넘음
round(vif_queue, 2) → 53.90
```

<details><summary>힌트</summary>

```text
접근방법:
- 각 변수를 나머지 변수들로 회귀한 보조 모형의 R^2 로 1/(1-R^2) 을 계산한다.

세부구현:
1. 변수 이름 네 개를 리스트에 담는다
2. 각 이름마다 나머지 이름을 ' + ' 로 이어 식을 만들어 ols 로 적합한다
3. 그 모형의 rsquared 로 VIF 를 계산해 모아 DataFrame 으로 만든다
4. 표에서 wait·queue 의 VIF 를 꺼내 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(vif_wait - 53.91) < 0.1
assert abs(vif_queue - 53.90) < 0.1
assert len(vif4) == 4
print("✅ 2단계 통과!")

### 3단계 — 겹치는 변수를 빼고 비교하기
**요구사항**:
- `queue` 를 뺀 3변수 모형 `smf.ols('sat ~ wait + talk + exp', data=cc).fit()` 을 `mod3` 에 담으세요.
- `mod3` 의 조정 결정계수를 `adj3`, 3변수 모형에서 `wait` 의 VIF 를 `vif_wait3` 에 담으세요.
- **1단계에서 담아 둔 `adj4`** 와 나란히 출력해, **설명력을 잃었는지** 한 줄로 함께 출력하세요.
- Adj R² 는 소수 **셋째 자리**, VIF 는 소수 **둘째 자리**까지 비교합니다.

**예시**
```
round(adj3, 3)      → 0.537    # 4변수 모형(0.537)과 사실상 같다
round(vif_wait3, 2) → 1.00     # 겹침이 사라졌다
```

<details><summary>힌트</summary>

```text
접근방법:
- queue 를 뺀 모형을 적합하고, 남은 세 변수로 VIF 를 다시 계산한다.

세부구현:
1. smf.ols 로 sat ~ wait + talk + exp 를 적합해 mod3 에 담는다
2. mod3.rsquared_adj 를 adj3 에 담는다
3. 세 변수 ['wait','talk','exp'] 로 VIF 표를 다시 만들어 wait 의 값을 꺼낸다
4. mod4 와 mod3 의 Adj R^2 를 나란히 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(adj3 - 0.5375) < 0.01
assert adj3 > adj4, '겹치는 변수를 빼면 조정 R^2 는 오히려 오릅니다'
assert abs(vif_wait3 - 1.00) < 0.05
print("✅ 3단계 통과!")

### 4단계 — 표준오차는 정말 √VIF 배 부풀려졌나
**배경**: 1부에서 "표준오차가 √VIF 배가 된다"고 했습니다. 방금 만든 두 모형으로 직접 확인합니다(3변수 모형의 `wait` VIF 가 1.00 이라 비교 기준으로 딱 맞습니다).

**요구사항**:
- `mod4` 와 `mod3` 에서 `wait` 계수의 표준오차(`.bse['wait']`)를 꺼내 비를 `se_ratio` 에 담으세요.
- `np.sqrt(vif_wait)` 를 `sqrt_vif` 에 담고, 두 값을 나란히 출력하세요.
- 두 값은 소수 **둘째 자리**까지 비교합니다.

**예시**
```
round(se_ratio, 2) → 7.34
round(sqrt_vif, 2) → 7.34    # 같은 값 — 이것이 '분산팽창'의 정확한 뜻
```

<details><summary>힌트</summary>

```text
접근방법:
- 두 모형의 표준오차를 나눠 배수를 구하고, VIF 의 제곱근과 비교한다.

세부구현:
1. mod4.bse['wait'] 를 mod3.bse['wait'] 로 나눠 se_ratio 에 담는다
2. np.sqrt(vif_wait) 를 sqrt_vif 에 담는다
3. 두 값을 소수 둘째 자리까지 나란히 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(se_ratio - 7.34) < 0.05
assert abs(se_ratio - sqrt_vif) < 0.05
print("✅ 4단계 통과!")

### 5단계 — 판단과 보고 (서술형)
**요구사항**: 아래 세 물음에 답하는 **보고 문장 3~4줄**을 `print` 로 출력하세요. 자가채점은 없습니다.

1. 네 변수 모형에서 대기고객수가 유의하지 않게 나온 **이유**는 무엇인가?
2. 어떤 처리를 했고, 처리 후 **무엇이 좋아졌으며 무엇을 잃었는가**?
3. 만약 목적이 "만족도를 **예측**하는 것"이었다면 같은 처리를 해야 했을까?

In [ ]:
# 여기에 코드를 작성하세요

---
## 실습 문제 2 — 펭귄: 내가 만든 파생변수가 공선성을 만들 때
**배경**: 펭귄의 **체중**을 물갈퀴 길이·부리 길이·부리 깊이로 설명하는 모형이 있습니다. 여기에 "부리 크기"를 한 번에 담는 **부리 면적(= 부리 길이 × 부리 깊이)** 을 파생변수로 추가하려 합니다. 추가하면 무슨 일이 생기고, 어떻게 다뤄야 할까요?

**최종 목표(자가채점 기준)**
| 단계 | 확인 항목 |
| --- | --- |
| 1단계 | 원래 세 변수 VIF — 최대 **2.673** (문제 없음) |
| 2단계 | 면적 추가 후 VIF — `bill_area` **227.10**, Adj R² **0.792** |
| 3단계 | 중심화 후 VIF — `area_c` **1.372**, 예측값 동일 |
| 4단계 | 판단 서술 (자가채점 없음) |

### 1단계 — 원래 모형 진단
**요구사항**:
- `data/penguins.csv` 를 읽고 `body_mass_g`·`flipper_length_mm`·`bill_length_mm`·`bill_depth_mm` **네 열의 결측 행을 제거**해 `pg_df` 에 담으세요(342행).
- 독립변수 세 개 `['flipper_length_mm', 'bill_length_mm', 'bill_depth_mm']` 의 VIF 를 구해 `vif_base`(DataFrame)에 담고 `display` 하세요.
- 가장 큰 VIF 를 `vif_max_base` 에 담아 출력하고, 기준선 5 와 비교해 한 줄로 판정하세요.
- VIF 는 소수 **셋째 자리**까지 비교합니다.

**예시**
```
len(pg_df)            → 342
round(vif_max_base, 3) → 2.673    # 5 미만 — 문제 없음
```

<details><summary>힌트</summary>

```text
접근방법:
- 네 열의 결측을 지운 뒤, 독립변수 세 개로 VIF 표를 만든다.

세부구현:
1. read_csv 후 dropna(subset=[네 열 이름]) 으로 결측 행을 제거한다
2. vif_table(데이터, [독립변수 세 개]) 로 표를 만든다
3. VIF 열의 최댓값을 max() 로 꺼내 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(pg_df) == 342
assert abs(vif_max_base - 2.673) < 0.01
assert len(vif_base) == 3
print("✅ 1단계 통과!")

### 2단계 — 파생변수(면적) 추가 후 진단
**요구사항**:
- `pg_df['bill_area'] = pg_df['bill_length_mm'] * pg_df['bill_depth_mm']` 로 파생변수를 만드세요.
- 네 변수(원래 셋 + `bill_area`)의 VIF 를 `vif_area` 에 담아 `display` 하고, `bill_area` 의 VIF 를 `vif_bill_area` 에 담으세요.
- 네 변수를 모두 넣은 회귀를 `mod_area` 에 담고 조정 결정계수를 `adj_area` 에 담아 출력하세요.
- VIF 는 소수 **둘째 자리**, Adj R² 는 소수 **셋째 자리**까지 비교합니다.

**예시**
```
round(vif_bill_area, 2) → 227.1    # 폭발
round(adj_area, 3)      → 0.792    # 그런데 설명력은 올랐다
```
> 계수의 유의성도 함께 출력해 보세요. **VIF 가 폭발했는데 계수는 전부 유의**하게 나오는, 혼란스러운 상황을 직접 보게 됩니다.

<details><summary>힌트</summary>

```text
접근방법:
- 곱셈으로 파생 열을 만들고, 네 변수로 VIF 표를 다시 만든 뒤 회귀를 적합한다.

세부구현:
1. 두 열을 곱해 bill_area 열을 만든다
2. vif_table 에 네 변수 리스트를 넘겨 표를 만든다
3. 표에서 bill_area 의 VIF 를 꺼낸다
4. smf.ols 로 네 변수를 모두 넣은 모형을 적합해 rsquared_adj 를 꺼낸다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(vif_bill_area - 227.10) < 0.5
assert abs(adj_area - 0.792) < 0.01
assert len(vif_area) == 4
print("✅ 2단계 통과!")

### 3단계 — 중심화로 처리하기
**요구사항**:
- 부리 길이·깊이에서 각각 **평균을 뺀** 열 `len_c`·`dep_c` 를 만들고, 그 둘을 곱해 `area_c` 를 만드세요 (원래 값을 곱한 뒤 중심화하는 것이 **아니라**, 중심화한 값끼리 곱합니다).
- `['flipper_length_mm', 'len_c', 'dep_c', 'area_c']` 의 VIF 를 `vif_cen` 에 담아 `display` 하고, `area_c` 의 VIF 를 `vif_area_c` 에 담으세요.
- 중심화 변수로 회귀를 적합해 `mod_cen` 에 담고, **두 모형의 예측값이 같은지** `np.allclose(mod_area.fittedvalues, mod_cen.fittedvalues)` 로 확인해 `same_pred` 에 담으세요.
- `mod_cen` 의 조정 R² 를 **2단계의 `adj_area`** 와 나란히 출력해 값이 같은지도 확인하세요.
- VIF 는 소수 **셋째 자리**까지 비교합니다.

**예시**
```
round(vif_area_c, 3) → 1.372    # 227.10 에서 내려왔다
same_pred            → True     # 예측은 전혀 달라지지 않았다
```

<details><summary>힌트</summary>

```text
접근방법:
- 두 변수에서 각각 평균을 뺀 열을 만들고, 그 둘을 곱해 상호작용 열을 만든다.
- 같은 구조의 회귀를 중심화 변수로 다시 적합해 예측값을 비교한다.

세부구현:
1. len_c = bill_length_mm - 그 열의 평균, dep_c 도 같은 방식으로 만든다
2. area_c = len_c * dep_c 로 만든다
3. vif_table 에 flipper 와 세 중심화 열을 넘겨 VIF 표를 만든다
4. smf.ols 로 body_mass_g ~ flipper_length_mm + len_c + dep_c + area_c 를 적합한다
5. np.allclose 로 두 모형의 fittedvalues 를 비교한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(vif_area_c - 1.372) < 0.01
assert same_pred
assert abs(mod_cen.rsquared_adj - adj_area) < 1e-6
print("✅ 3단계 통과!")

### 4단계 — 판단 (서술형)
**요구사항**: 아래 두 물음에 답하는 **3줄 내외**의 결론을 `print` 로 출력하세요. 자가채점은 없습니다.

1. 부리 면적을 모형에 **넣는 것 자체가 잘못**이었나? 근거와 함께 답하세요.
2. 이 모형으로 "부리 길이가 체중에 미치는 영향"을 보고해야 한다면, 어떤 버전을 쓰고 그 이유는 무엇인가?

In [ ]:
# 여기에 코드를 작성하세요

---
## 정리

| 질문 | 답 |
|---|---|
| 왜 생기나 | 원래 겹치는 변수 · 파생/합계 변수 · 제곱항·상호작용항 · 더미 함정 |
| 무엇이 망가지나 | **계수의 표준오차가 √VIF 배로 팽창** → 신뢰구간·p 가 커지고 계수가 흔들림 |
| 무엇은 멀쩡한가 | **예측값·R²** — 그래서 목적이 예측이면 손대지 않아도 된다 |
| 어떻게 재나 | 상관행렬(빠르지만 짝만 봄) → **VIF**(표준 도구, 5 경고선·10 컷오프) → `Cond. No.`(참고) |
| 어떻게 고치나 | 목적 확인 → **제거** · **합성** · **중심화**(구조적 공선성) · 표본 확보 · 정규화 회귀 |
| 마지막에 할 일 | 처리 전후의 **VIF·계수·Adj R² 를 나란히** 놓고 확인하고, 그 과정을 리포트에 적는다 |

다중공선성은 "에러가 나는 문제"가 아니라 **결과를 조용히 오해하게 만드는 문제**입니다. 다중회귀를 돌렸는데 **혼자서는 강했던 변수가 갑자기 유의하지 않다면**, 가장 먼저 의심해야 할 후보입니다.